**ISD-1020 · Master ISD (apprenticeship) · Université Paris-Saclay**

**Before any edit:** `File → Save a copy in Drive`.
Work only on **your** copy.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, torch

print("sklearn", sklearn.__version__, "| torch", torch.__version__)

# Course helpers — skip this

This cell is **not** an exercise: it downloads the plotting helpers and dataset
loaders if they are not already next to the notebook. You do not need to read
it. Continue below.


In [ ]:
# Environment setup (when run outside the repository, e.g. Google Colab)
import sys
import urllib.request
from pathlib import Path

_RAW_BASE = "https://raw.githubusercontent.com/stephane-rivaud/M2-ISD-ML-DL/main"

for _start in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_start / "pyproject.toml").is_file() and (_start / "common").is_dir():
        if str(_start) not in sys.path:
            sys.path.insert(0, str(_start))
        break

try:
    for module_path in ("common/__init__.py", "common/plots.py", "data/fetch.py"):
        local_file = Path(module_path)
        if local_file.is_file() or any(
            (Path(p) / module_path).is_file() for p in sys.path if p
        ):
            continue
        local_file.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{_RAW_BASE}/{module_path}", local_file)
except Exception:
    print(
        "Could not download the course helpers. Clone or download this repository so common/ and data/ sit next to this notebook."
    )


# Tables and boosting — which machine will fail?

ISD-1020 · Monday 21 September 2026 · morning workshop (~140 min after the
40 min lecture, including ~10 min of optional fast track).

Thread of the module: **1. Problem · 2. Data · 3. Baseline · 4. Model · 5. Ablation ·
6. Error analysis → Recommendation**.
On 14 September: Telco churn. Today: machine failures, metrics
for strong imbalance, leakages, boosting.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from common.plots import plot_confusion, plot_pr_curve
from data.fetch import load_ai4i

RANDOM_STATE = 0
TARGET = "Machine failure"
ID_COLS = ["UDI", "Product ID"]
FAILURE_FLAGS = ["TWF", "HDF", "PWF", "OSF", "RNF"]
COST_FN = 100  # relative cost of a missed failure (unplanned stop)
COST_FP = 1  # relative cost of a false alarm (extra inspection)


def decision_cost(y_true, y_pred, cost_fn=COST_FN, cost_fp=COST_FP):
    """Return (total_cost, n_false_negatives, n_false_positives)."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n_fn = int(((y_true == 1) & (y_pred == 0)).sum())
    n_fp = int(((y_true == 0) & (y_pred == 1)).sum())
    return n_fn * cost_fn + n_fp * cost_fp, n_fn, n_fp

## The problem — ⏱ ~8 min

A fleet of machines (a machining line): at each cycle we observe
sensors, and we want to know whether **this machine will fail**
(`Machine failure` = 1). Scheduling maintenance costs less
than an unplanned stop.

The costly error is the **false negative** (real failure, silent model).
Always answering "no failure" is almost always true — and
**useless**.

In pairs (30 s): what concrete decision would a line manager
take with a list of failure scores?

## Data — ⏱ ~10 min

AI4I 2020 Predictive Maintenance (UCI): 10,000 cycles, one row =
one cycle. Local copy if it is there (`data/ai4i.parquet`), otherwise UCI.

- `Type`: product quality **L / M / H**.
- Sensors: air and process temperatures, speed, torque, tool wear.
- Target: `Machine failure`.
- Five failure-mode **sub-flags**: `TWF` (tool wear), `HDF`
  (heat dissipation), `PWF` (power), `OSF` (overstrain),
  `RNF` (random). These are **not** sensors available before
  the failure: they are after-the-fact diagnostics.

In [ ]:
df = load_ai4i()
print("shape:", df.shape)
print("columns:", list(df.columns))
df.head()

In [ ]:
df.describe()

In [ ]:
print("Missing values:")
print(df.isna().sum())
print()
print("Type (quality):")
print(df["Type"].value_counts())

**Exercise.** Compute the **failure rate**: the share of rows with
`Machine failure == 1`. Also print `value_counts()` of the five
sub-flags `TWF`, `HDF`, `PWF`, `OSF`, `RNF`.
You should find **3.390%** failures (339 / 10,000).

In [ ]:
# To complete
...

339 failures, of which 330 have at least one `TWF`/`HDF`/`PWF`/`OSF` flag.
Nine failures have **none** of these four flags. `RNF` (19 rows) almost
never coincides with the target: it is not "the target in
pieces", but it looks enough like it to **leak**.

## Metrics for imbalance — ⏱ ~12 min

Accuracy = share of correct answers. With 3.4% failures, "always 0"
already scores ~96.6%.

| Name | Formula (class 1 = failure) | Reads |
|---|---|---|
| **Precision** | TP / (TP + FP) | Among the alarms, what share is true? |
| **Recall** | TP / (TP + FN) | Among the failures, what share did we see? |
| **F1** | harmonic mean of the two | Compromise, at threshold 0.5 |
| **PR-AUC** | area under the precision–recall curve | Quality of the **score**, all thresholds |

The no-skill PR-AUC equals the positive rate (here 0.0339): a horizontal
line at 3.4% precision.

**Exercise.** On the whole `df`, predict **always 0**. Compute
accuracy, precision, recall, F1 (`zero_division=0`) and the no-skill
PR-AUC `y.mean()`. Keep these numbers in view.

In [ ]:
# To complete
...

**Accuracy lies.** 96.6% correct answers **without ever catching
a failure**. A model at "97% accuracy" may be only a step
above "always no". We will read **recall, precision, F1,
PR-AUC**, not accuracy alone.

## Pipeline and ColumnTransformer — ⏱ ~15 min

**Legitimate** features: `Type` + the five sensors. We **drop**
`UDI`, `Product ID` (unique identifiers) and the five sub-flags
(leakage, next section).

- numeric → `StandardScaler` **inside** the `Pipeline` (fit on
  train only);
- `Type` → `OneHotEncoder`;
- **stratified** 80/20 split, `random_state=RANDOM_STATE`.

The `Pipeline` is already built. Your turn: the split, then `fit`.

In [ ]:
y = df[TARGET].copy()
X = df.drop(columns=ID_COLS + [TARGET] + FAILURE_FLAGS)
NUMERIC = [c for c in X.columns if c != "Type"]
CATEGORICAL = ["Type"]

preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), NUMERIC),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            CATEGORICAL,
        ),
    ]
)
pipe = Pipeline(
    [
        ("prep", preprocessor),
        (
            "clf",
            LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        ),
    ]
)
print("features:", list(X.columns))
print("numeric:", NUMERIC)
print("categorical:", CATEGORICAL)

**Exercise.** Split `X`, `y` into train/test: `test_size=0.2`,
`stratify=y`, `random_state=RANDOM_STATE`. Expected names:
`X_train`, `X_test`, `y_train`, `y_test`.

In [ ]:
# To complete
...

In [ ]:
print("train:", X_train.shape, "test:", X_test.shape)
print("failure rate train:", float(y_train.mean()))
print("failure rate test: ", float(y_test.mean()))
print("positives in test:", int(y_test.sum()))

**Exercise.** Fit the pipeline: `pipe.fit(X_train, y_train)`.

In [ ]:
# To complete
...

In [ ]:
y_pred = pipe.predict(X_test)
y_score = pipe.predict_proba(X_test)[:, 1]
print(f"model accuracy   = {accuracy_score(y_test, y_pred):.3f}")
print(f"majority on test = {accuracy_score(y_test, np.zeros_like(y_test)):.3f}")
print(f"predicted positives: {int(y_pred.sum())}  (true positives in test: {int(y_test.sum())})")
plot_confusion(y_test, y_pred, labels=[0, 1])
plt.show()
plot_pr_curve(y_test, y_score)
plt.show()

Checkpoint (⏱ ~5 min, Solal circulating): every group has a fitted
`pipe`, a confusion matrix and a PR curve. The cell **true 1,
predicted 0** is the pile of missed failures.

**Exercise.** On the test set, compute precision, recall, F1
(`zero_division=0`) and PR-AUC (`average_precision_score` on
`y_test`, `y_score`). Compare recall to 0 (baseline) and PR-AUC
to the no-skill `y_test.mean()`.

In [ ]:
# To complete
...

**Take-away.** Baseline: accuracy 0.966, recall **0**, no-skill PR-AUC
0.034. Logistic: accuracy 0.975 (looks like little) but recall 0.279
and PR-AUC 0.504. We start catching failures. **Read recall
and PR-AUC, not accuracy.**

## Cost matrix — ⏱ ~10 min

A single number for a **decision**. Pedagogical example (**relative**
costs, not a factory invoice):

- false negative = unplanned stop → `COST_FN = 100`
- false positive = useless inspection → `COST_FP = 1`

`decision_cost(y_true, y_pred)` (already defined) returns the total cost
on a batch, plus the FN and FP counts.

**Exercise.** On the **test** set, cost of "always 0" and cost of the
logistic (`y_pred`). Print cost, FN, FP for both.

In [ ]:
# To complete
...

On this split: 68 failures in the test set. Always 0 costs **6,800**
(68 × 100). The logistic misses 49 failures and raises 2 false alarms
→ **4,902**. Better than the baseline, far from enough: recall
0.28 leaves too many unplanned stops.

## Two leakages — ⏱ ~15 min

Leakage is information from the **test** set (or from the **target**) that
was used to prepare the model. The score lies; in production it drops.

### Leakage 1 — scaler before the split

**Exercise.** Wrong order: `StandardScaler` on **all** of `X`, then
split, then logistic **without** re-scaling (`passthrough` on the
numeric columns). Print this version's PR-AUC and that of the honest
`pipe` (already computed: `model_pr_auc`).

In [ ]:
# To complete
...

Here the PR-AUC gap is **negligible** (10,000 rows: the train mean
≈ the global mean). That is not permission: the leakage
is **silent**. The remedy is not "see whether the score moves",
it is the `Pipeline`.

### Leakage 2 — the target sub-flags

**Exercise.** Take `Type` + sensors **and** `TWF`, `HDF`, `PWF`,
`OSF`, `RNF`. Same split, same logistic in a `Pipeline`. Print
recall and PR-AUC, to compare with 0.279 / 0.504.

In [ ]:
# To complete
...

Recall **0.941** and PR-AUC **0.945**: the model "read" the post-failure
diagnosis. In production these flags do not exist yet. It is not
1.0: nine failures have no flag — even a leakage leaves a
residue. **Never put the target, or its pieces, into `X`.**

## Cross-validation — ⏱ ~10 min

A single 80/20 split is noisy (68 failures in the test set). We compare
models by **PR-AUC** in `StratifiedKFold` **on the train set**
(the test set stays intact for the final number).

Five folds, `shuffle=True`, `random_state=RANDOM_STATE`.
`cross_val_score` **clones** the pipeline: no leakage between folds.

**Exercise.** `StratifiedKFold(n_splits=5, shuffle=True,
random_state=RANDOM_STATE)`, puis `cross_val_score` du `pipe`
logistique sur `X_train`, `y_train`, `scoring="average_precision"`.
Print the five scores, the mean and the standard deviation.

In [ ]:
# To complete
...

## Boosting vs logistic — ⏱ ~12 min

Same preprocessor, a `HistGradientBoostingClassifier`: trees
added one after another to correct the **residuals** of the previous one.
This is the module's **tabular baseline** (to beat, not to decorate).
`max_iter=80` to stay within the CPU budget.

In [ ]:
pipe_hgb = Pipeline(
    [
        ("prep", preprocessor),
        (
            "clf",
            HistGradientBoostingClassifier(max_iter=80, random_state=RANDOM_STATE),
        ),
    ]
)

**Exercise.** `pipe_hgb.fit(X_train, y_train)`. On the test set: recall,
F1, PR-AUC. Then `cross_val_score` as above (same `cv` and
`scoring`). Compare with the logistic.

In [ ]:
# To complete
...

In [ ]:
print(f"HGB accuracy = {accuracy_score(y_test, y_hgb):.3f}")
cost_hgb, fn_hgb, fp_hgb = decision_cost(y_test, y_hgb)
print(f"HGB cost     = {cost_hgb}   FN={fn_hgb}   FP={fp_hgb}")
plot_confusion(y_test, y_hgb, labels=[0, 1])
plt.show()
plot_pr_curve(y_test, score_hgb)
plt.show()

**Take-away.** HGB: recall 0.662, PR-AUC 0.805, cost 2,307 (23 FN,
7 FP) versus 4,902 (logistic) and 6,800 (baseline). Train CV: HGB
0.804 ± 0.017 vs logistic 0.438 ± 0.040. On this table, boosting
**is** the model to beat — not a network, for now.

## Permutation importance — ⏱ ~8 min

Shuffle one test column, measure the **drop in PR-AUC**.
This is not a regression weight: it works for boosting.
`n_repeats=5` (not 30) for the CPU budget.

In [ ]:
perm = permutation_importance(
    pipe_hgb,
    X_test,
    y_test,
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=1,
    scoring="average_precision",
)
order = perm.importances_mean.argsort()
print("Permutation importance (drop in PR-AUC):")
for i in order[::-1]:
    print(
        f"  {X_test.columns[i]:28s}  "
        f"{perm.importances_mean[i]:.3f} ± {perm.importances_std[i]:.3f}"
    )
fig, ax = plt.subplots()
ax.barh(X_test.columns[order], perm.importances_mean[order])
ax.set_xlabel("Drop in PR-AUC (permutation, 5 repeats)")
ax.set_title("Permutation importance")
plt.show()

On this split, **torque** (`Torque`) dominates, then the temperatures,
speed and tool wear. `Type` weighs little for the global score —
that is not a reason to ignore errors **by** type.

## Errors by product type — ⏱ ~10 min

A global recall of 0.66 can hide a segment. `Type` L / M / H does not
have the same failure rate (L higher) nor the same volume (H is rare).

**Exercise.** For each `Type` in `{"L", "M", "H"}` on the **test** set,
print the number of rows, the number of true failures, HGB recall
(`y_hgb`) and the number of false negatives.

In [ ]:
# To complete
...

On this split: L (40 failures) recall 0.675; M (21) 0.762; **H (7)
0.286**. Seven high-quality failures is few — do not over-interpret
a percentage. On the other hand: a global score does not say whether
the rarer H segment is protected. Before deploying: **does a single
threshold suit all three qualities?**

## Fast track — ⏱ ~10 min (optional)

In [ ]:
# ⚡ Fast track (optional) — for those who have already done ML
pipe_balanced = Pipeline(
    [
        ("prep", preprocessor),
        (
            "clf",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
pipe_balanced.fit(X_train, y_train)
pred_b = pipe_balanced.predict(X_test)
score_b = pipe_balanced.predict_proba(X_test)[:, 1]
cost_b, fn_b, fp_b = decision_cost(y_test, pred_b)
print("class_weight='balanced' logistic")
print(f"  recall = {recall_score(y_test, pred_b):.3f}")
print(f"  prec   = {precision_score(y_test, pred_b, zero_division=0):.3f}")
print(f"  PR-AUC = {average_precision_score(y_test, score_b):.3f}")
print(f"  cost   = {cost_b}   FN={fn_b}   FP={fp_b}")
print("HGB still wins PR-AUC; balanced logistic buys recall with many FP.")

In [ ]:
# ⚡ Fast track (optional) — for those who have already done ML
# Threshold sweep on the fitted HGB: expected cost vs decision threshold.
thresholds = np.linspace(0.05, 0.95, 19)
rows = []
for threshold in thresholds:
    pred_t = (score_hgb >= threshold).astype(int)
    cost_t, n_fn_t, n_fp_t = decision_cost(y_test, pred_t)
    rows.append((threshold, cost_t, n_fn_t, n_fp_t, pred_t.mean()))
cost_table = pd.DataFrame(
    rows, columns=["threshold", "cost", "FN", "FP", "alarm_rate"]
)
best = cost_table.loc[cost_table["cost"].idxmin()]
print(cost_table.round(3).to_string(index=False))
print(
    f"min cost = {int(best['cost'])} at threshold = {best['threshold']:.2f} "
    f"(default 0.5 cost {cost_hgb})"
)

## Attributions

- **AI4I 2020 Predictive Maintenance**: S. Matzka / UCI repository
  id 601, licence [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
  Record:
  [AI4I 2020](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset).
- **Pipeline, validation, metrics**: [MOOC scikit-learn (Inria)](https://inria.github.io/scikit-learn-mooc/),
  modules 1 (*The predictive modeling pipeline*), 2 (*Selecting the best
  model*) and 7 (*Evaluating model performance*), licence
  [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

## To go further at home

1. Module 1 — *The predictive modeling pipeline* :
   [overview](https://inria.github.io/scikit-learn-mooc/predictive_modeling_pipeline/predictive_modeling_module_intro.html)
2. Module 2 — *Selecting the best model* :
   [overview](https://inria.github.io/scikit-learn-mooc/overfit/overfit_module_intro.html)
3. Module 7 — *Evaluating model performance* :
   [overview](https://inria.github.io/scikit-learn-mooc/evaluation/evaluation_module_intro.html)

## Empirical protocol — ⏱ ~15 min

The six cells below are the thread of the module (and of the oral exam).
Answer in prose, tied to *this* notebook.

## Problem

Answer in prose (not only code).

1. Which **business decision** should the model inform, and what is the **target**?
2. Which **metric** matches that decision, and why not accuracy alone?
3. How do you separate training and test **without leakage** (split unit, time, stratification)?

## Data

Answer in prose (not only code).

1. What are the variables, their types, and any notable **imbalances** or volumes?
2. Which columns must you **not** use (identifiers, leakages, target sub-flags)?
3. What do you know about **quality** (missing values, outliers, drift) and temporal or business coverage?

## Baseline

Answer in prose (not only code).

1. Which **naive baseline** (majority class, mean, seasonal) and what score does it get?
2. Why is this baseline the **honest floor**, and not a "deliberately weak" model?
3. Which score must you **beat** to justify a more complex model?

## Model

Answer in prose (not only code).

1. Which **model family** do you choose, and why (not "because it is deep")?
2. How do you train it (split, validation, hyperparameters, loss)?
3. Does the model **beat the baseline** on the chosen metric, on an uncontaminated split?

## Ablation

Answer in prose (not only code).

1. What happens if you remove a family of variables, a regulariser, or a block of the network?
2. Which choice (features, architecture, horizon, threshold) **actually changes** the score?
3. Does the gain justify the extra **complexity** relative to the baseline?

## Error analysis → Recommendation

Answer in prose (not only code).

1. Where does the model go wrong (**segments**, error types, horizon)?
2. Are these errors **costly** for the business, and what would you change in the data or the model?
3. What concrete **recommendation**: deploy, do not deploy, stay on the baseline, or collect this specific data?